# 🪲 MuseumSCAT — Notebook 01: EDA + Baseline

> *"Prima si guarda il nemico, poi si carica il fucile."* — proverbio da Grandmaster
> inventato cinque minuti fa.

Benvenuto nella **collezione di scarabei stercorari danesi** del Natural History
Museum of Denmark. Il nostro compito **non** è riconoscere gli scarabei: è
**trascrivere due campi** scritti sui cartellini di ogni esemplare, esattamente
come li ha vergati un entomologo tra il tardo '800 e oggi.

I due campi da predire per ogni immagine:

| Campo | Cosa contiene | Esempi |
|---|---|---|
| `verbatimDate` | la data di raccolta, *verbatim* | `27.IV.2022`, `18-7 1965`, `IV.91`, `MISSING` |
| `verbatimLocality` | il luogo di raccolta, *verbatim* | `Dyrehaven`, `Kb \| Dyrehaven`, `MISSING` |

**I numeri che contano (e spaventano):**
- **200** esempi etichettati (`train.csv`) contro **~3.300** immagini di test. Pochissimo
  training → niente OCR addestrato da zero: si va di modelli **pre-addestrati** + few-shot.
- Metrica **doppia**: **NED** (Normalized Edit Distance) **+ AURC**. Non basta trascrivere
  bene: devi anche dare un **confidence score** onesto per ogni predizione. Il modello
  deve "sapere quello che non sa".

**Piano di questo notebook:**
1. Trovare e caricare i dati (regola 1: non fidarsi mai dei path).
2. EDA sulle etichette: formati delle date, delle località, valori mancanti, cartellini multipli.
3. Guardare le immagini vere.
4. Reimplementare la metrica NED in locale → così possiamo validarci **senza** sprecare submission.
5. Una **baseline naive** che produce un `submission.csv` valido: numero sulla leaderboard, subito.
6. Cosa ci ha insegnato l'EDA → e quale modello scegliere nel Notebook 02.

Andiamo. 🚀

## 1. Setup e caricamento dati

Regola numero 1 del mestiere: **non scrivere mai a mano il path dei dati.** Kaggle
monta la competizione sotto `/kaggle/input/<slug>/`, ma lo slug può cambiare. Cerchiamolo.

In [ ]:
import os, glob, re, itertools, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

pd.set_option('display.max_colwidth', 120)
random.seed(42); np.random.seed(42)

# Path della competizione su Kaggle (stabile: niente auto-discovery)
DATA_DIR = '/kaggle/input/competitions/museumscat-specimen-collection-annotation-task'
IMG_DIR  = os.path.join(DATA_DIR, 'images')

print('DATA_DIR =', DATA_DIR)
print('IMG_DIR  =', IMG_DIR)
print('Contenuto:')
for f in sorted(os.listdir(DATA_DIR)):
    print('  ', f)

In [ ]:
print('Numero immagini (jpeg):', len(glob.glob(os.path.join(IMG_DIR, '*.jpeg'))))

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print('train:', train.shape)
print('test :', test.shape)
print('\nColonne train:', list(train.columns))
print('Colonne test :', list(test.columns))
train.head()

### 1.1 Un piccolo mistero: `NULL` o `MISSING`?

La documentazione della gara si contraddice: in un punto dice di segnare i valori
mancanti come `"NULL"`, in un altro come `"MISSING"`. **Non indovinare: guarda i dati.**
Questo è un riflesso che devi allenare — la ground truth è l'unica fonte di verità.

In [ ]:
for col in ['verbatimDate', 'verbatimLocality']:
    print(f'--- {col} ---')
    vc = train[col].astype(str).value_counts()
    # mostra i token "sospetti" di mancanza
    for tok in ['MISSING', 'NULL', 'nan', 'None', '']:
        if tok in vc.index:
            print(f'  token "{tok}": {vc[tok]} occorrenze')
    print('  valori nulli pandas (NaN):', train[col].isna().sum())
    print()

# Definiamo il token di "mancante" REALE trovato nei dati (non quello della doc)
MISSING_TOKEN = 'MISSING'
for tok in ['MISSING', 'NULL']:
    if (train[['verbatimDate','verbatimLocality']].astype(str) == tok).any().any():
        MISSING_TOKEN = tok
        break
print('>>> Token di mancanza usato nella ground truth:', repr(MISSING_TOKEN))

## 2. EDA sulle etichette

Prima ancora delle immagini, i **testi** ci raccontano moltissimo: quanto sono lunghi,
quanto spesso mancano, quali formati assumono. Da qui capiremo *quanto è difficile* il
problema e *dove* un modello sbaglierà.

In [ ]:
# Frequenza dei valori mancanti in ciascun campo
for col in ['verbatimDate', 'verbatimLocality']:
    s = train[col].astype(str)
    n_missing = (s == MISSING_TOKEN).sum()
    print(f'{col:20s}  mancanti: {n_missing:3d} / {len(s)}  ({100*n_missing/len(s):.1f}%)')

In [ ]:
# Distribuzione della lunghezza (in caratteri) dei campi NON mancanti
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['verbatimDate', 'verbatimLocality']):
    s = train[col].astype(str)
    lengths = s[s != MISSING_TOKEN].str.len()
    ax.hist(lengths, bins=20, color='#008ABC', edgecolor='white')
    ax.set_title(f'Lunghezza di {col}')
    ax.set_xlabel('caratteri'); ax.set_ylabel('conteggio')
plt.tight_layout(); plt.show()

### 2.1 Che aspetto hanno le date?

La doc ci avvisa di un delirio di formati: numeri romani per il mese (`IV` = aprile),
date parziali (`IV.91`), separatori misti. Classifichiamo automaticamente ogni data in
una "famiglia" di formato con qualche regex. Non serve la perfezione: serve capire la
distribuzione.

In [ ]:
ROMAN = r'(?:I{1,3}|IV|V|VI{0,3}|IX|XI{0,2}|X)'

def date_family(s):
    s = str(s).strip()
    if s == MISSING_TOKEN:
        return 'missing'
    if re.search(ROMAN, s):                       # mese in numeri romani
        return 'roman_numeral'
    if re.search(r'\d{1,2}[./\- ]\d{1,2}[./\- ]\d{2,4}', s):  # gg sep mm sep aaaa
        return 'numeric_full'
    if re.search(r'\d{1,2}[./ ]\d{2,4}$', s):     # parziale tipo mese/anno
        return 'partial'
    if re.search(r'\d', s):
        return 'other_numeric'
    return 'no_digits'

fam = train['verbatimDate'].map(date_family)
print(fam.value_counts())
fam.value_counts().plot.bar(color='#008ABC', figsize=(8,3), title='Famiglie di formato data')
plt.tight_layout(); plt.show()

### 2.2 Cartellini multipli: il trucco della pipe `|`

Alcuni esemplari hanno le informazioni spalmate su **più cartellini** fisici sullo
stesso spillo. Nella ground truth vengono uniti con `|`. La metrica prova **tutti gli
ordinamenti** dei cartellini e poi rimuove le pipe: quindi puoi usarle generosamente
quando l'ordine di lettura è ambiguo. Vediamo quanto sono frequenti.

In [ ]:
for col in ['verbatimDate', 'verbatimLocality']:
    s = train[col].astype(str)
    has_pipe = s.str.contains(r'\|')
    print(f'{col:20s}  con pipe (multi-card): {has_pipe.sum()} ({100*has_pipe.mean():.1f}%)')

print('\nEsempi multi-card:')
mask = train['verbatimLocality'].astype(str).str.contains(r'\|')
display(train.loc[mask, ['image_file','verbatimDate','verbatimLocality']].head(8))

### 2.3 Località: coordinate, gerarchie e le trappole (`Dania`, sterco di mucca)

Le località vanno dal semplice `Dyrehaven` a stringhe con coordinate GPS. Ricorda le
due trappole della doc: **`Dania`** (nome della collezione, non un luogo) e **`i kogøding`**
(= "nello sterco di mucca", non un luogo) **non** compaiono nella ground truth. Controlliamo.

In [ ]:
loc = train['verbatimLocality'].astype(str)

has_coord = loc.str.contains(r'\d+\.\d+\s*[NnSs].*\d+\.\d+\s*[EeWw]', regex=True)
print('Località con coordinate GPS:', has_coord.sum())

# Verifica trappole: NON dovrebbero comparire come località
for trap in ['Dania', 'kogøding', 'kogjød', 'Coll.', 'det.', 'Tilg.']:
    n = loc.str.contains(re.escape(trap), case=False).sum()
    print(f'  contiene "{trap}": {n}')

print('\nLocalità più frequenti (non mancanti):')
print(loc[loc != MISSING_TOKEN].value_counts().head(12))

## 3. Guardiamo le immagini vere

I numeri sono belli ma il problema è **visivo**. Mostriamo alcuni esemplari con la loro
ground truth accanto: è qui che capisci quanto è dura la scrittura a mano storica, e
perché un semplice OCR soffrirà.

In [ ]:
def load_image(image_file):
    for base in [IMG_DIR, DATA_DIR]:
        p = os.path.join(base, image_file)
        if os.path.exists(p):
            return Image.open(p).convert('RGB')
    hits = glob.glob(os.path.join(IMG_DIR, '**', image_file), recursive=True)
    return Image.open(hits[0]).convert('RGB') if hits else None

sample = train.sample(min(6, len(train)), random_state=1).reset_index(drop=True)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = load_image(row['image_file'])
    if img is not None:
        ax.imshow(img)
    ax.set_title(f"date: {row['verbatimDate']}\nloc: {row['verbatimLocality']}", fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 4. La metrica in casa nostra: NED (e un assaggio di AURC)

**Questa è la parte più importante di tutto il notebook.** Su Kaggle hai poche submission
al giorno: se ti validi solo sulla leaderboard, sei cieco. Reimplementiamo la metrica in
locale, così possiamo misurare *qualsiasi* idea sui 200 esempi di train prima di sprecare
un tentativo.

La **NED** (Normalized Edit Distance) tra predizione e verità:
- `0.0` = match perfetto, `1.0` = completamente sbagliato;
- **case-insensitive**;
- per le **date**, la punteggiatura è normalizzata (`.  ,  -  ·  spazio` sono equivalenti);
- gestisce le **pipe**: prova tutti gli ordinamenti dei cartellini e tiene il migliore.

> ⚠️ È la *nostra* approssimazione della metrica ufficiale: utile per il CV locale, non
> identica al decimale. Ma per confrontare due idee è più che sufficiente.

In [ ]:
def levenshtein(a, b):
    "Distanza di edit classica (pura Python: le stringhe sono corte, va benissimo)."
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def normalize(s, is_date):
    s = str(s).strip().lower()
    if is_date:
        s = re.sub(r'[\.,\-·/\s]+', ' ', s).strip()  # unifica separatori di data
    return s

def ned_single(pred, truth, is_date):
    "NED con gestione pipe: prova ogni ordinamento dei cartellini della verità."
    p_cards = [normalize(x, is_date) for x in str(pred).split('|')]
    t_cards = [normalize(x, is_date) for x in str(truth).split('|')]
    best = 1.0
    # prova tutte le permutazioni dei cartellini di verità contro quelli predetti
    for perm in itertools.permutations(t_cards):
        p_join = ' '.join(p_cards)
        t_join = ' '.join(perm)
        d = levenshtein(p_join, t_join)
        m = max(len(p_join), len(t_join), 1)
        best = min(best, d / m)
    return best

# sanity check
assert ned_single('27.5.2022', '27-5-2022', is_date=True) == 0.0
assert ned_single('Dyrehaven', 'dyrehaven', is_date=False) == 0.0
print('NED sanity check: OK ✅')

In [ ]:
def score_df(pred_df, truth_df):
    "NED medio sui due campi, come proxy dello score (piu basso = meglio)."
    m = truth_df.merge(pred_df, on='image_file', suffixes=('_true', '_pred'))
    date_ned = m.apply(lambda r: ned_single(r['verbatimDate_pred'],  r['verbatimDate_true'],  True),  axis=1)
    loc_ned  = m.apply(lambda r: ned_single(r['verbatimLocality_pred'], r['verbatimLocality_true'], False), axis=1)
    return date_ned.mean(), loc_ned.mean(), (date_ned.mean() + loc_ned.mean()) / 2

### 4.1 Un assaggio di AURC (perché la confidence conta)

L'**AURC** ordina le predizioni per confidence decrescente e misura la NED media man mano
che "copri" sempre più predizioni. L'idea: un buon modello è **sicuro quando ha ragione**
e **incerto quando sbaglia**. Alta confidence su risposte sbagliate = punizione severa.

Non replichiamo la formula ufficiale al decimale, ma implementiamo la logica per capirla.

In [ ]:
def aurc(neds, confidences):
    "Area under risk-coverage curve (approssimazione). neds e confidences: array 1D."
    order = np.argsort(-np.asarray(confidences))      # confidence alta prima
    neds_sorted = np.asarray(neds)[order]
    coverages = np.arange(1, len(neds_sorted) + 1)
    risk = np.cumsum(neds_sorted) / coverages          # NED media cumulata
    return risk.mean()                                  # area (media) sotto la curva

# Intuizione: con la STESSA qualita di predizioni, ordinare bene la confidence abbassa l'AURC.
demo_neds = np.array([0.0, 0.0, 1.0, 1.0])
print('Confidence perfetta (giuste prima):', round(aurc(demo_neds, [1,1,0,0]), 3))
print('Confidence pessima  (sbagliate prima):', round(aurc(demo_neds, [0,0,1,1]), 3))

## 5. Baseline naive → prima submission

Obiettivo di una baseline: **non vincere**, ma (a) validare la pipeline end-to-end e
(b) mettere un numero sulla leaderboard come riferimento. Confrontiamo due strategie
banali sui 200 esempi di train con la *nostra* NED:

1. **Tutto `MISSING`** — sorprendentemente competitiva se molti campi sono davvero vuoti.
2. **Valore più frequente** — la "moda" di date e località.

Poi scegliamo la migliore e generiamo il `submission.csv`.

In [ ]:
# Split interno per una valutazione onesta (train piccolo: teniamo l'80/20)
tr = train.sample(frac=0.8, random_state=0)
va = train.drop(tr.index)
truth_va = va[['image_file','verbatimDate','verbatimLocality']].copy()

most_common_date = tr['verbatimDate'].astype(str).mode()[0]
most_common_loc  = tr['verbatimLocality'].astype(str).mode()[0]
print('Data piu frequente     :', repr(most_common_date))
print('Localita piu frequente :', repr(most_common_loc))

def make_pred(df, date_val, loc_val):
    return pd.DataFrame({
        'image_file': df['image_file'].values,
        'verbatimDate': date_val,
        'verbatimLocality': loc_val,
    })

strategies = {
    'tutto MISSING'    : make_pred(va, MISSING_TOKEN, MISSING_TOKEN),
    'valore piu freq.' : make_pred(va, most_common_date, most_common_loc),
}
print('\n%-18s  %8s  %8s  %8s' % ('strategia','NED date','NED loc','NED avg'))
scores = {}
for name, pred in strategies.items():
    d, l, a = score_df(pred, truth_va)
    scores[name] = a
    print('%-18s  %8.3f  %8.3f  %8.3f' % (name, d, l, a))

best_strategy = min(scores, key=scores.get)
print('\n>>> Baseline scelta:', best_strategy)

### 5.1 Costruzione del `submission.csv`

Il formato di submission atteso (5 colonne):

`image_file, verbatimDate, verbatimDate_confidence, verbatimLocality, verbatimLocality_confidence`

Due note pratiche:
- Kaggle **non accetta celle vuote**: i mancanti vanno scritti col token (`MISSING`).
- La **confidence** serve all'AURC. Per una baseline uniforme la mettiamo costante (es. `0.5`):
  non aiuta il ranking, ma è un valore valido. Il vero lavoro sulla confidence arriva col modello.

In [ ]:
# Applica la strategia migliore a TUTTO il test set
if best_strategy == 'tutto MISSING':
    date_val, loc_val = MISSING_TOKEN, MISSING_TOKEN
else:
    date_val = train['verbatimDate'].astype(str).mode()[0]
    loc_val  = train['verbatimLocality'].astype(str).mode()[0]

submission = pd.DataFrame({
    'image_file': test['image_file'].values,
    'verbatimDate': date_val,
    'verbatimDate_confidence': 0.5,
    'verbatimLocality': loc_val,
    'verbatimLocality_confidence': 0.5,
})

# Controlli di validità: niente NaN, niente celle vuote, tutte le immagini di test presenti
assert submission['image_file'].nunique() == test['image_file'].nunique()
assert not submission.isna().any().any(), 'ci sono NaN!'
assert (submission[['verbatimDate','verbatimLocality']] != '').all().all(), 'celle vuote!'

submission.to_csv('submission.csv', index=False)
print('submission.csv scritto:', submission.shape)
submission.head()

## 6. Cosa ci ha insegnato l'EDA (e come scegliamo il modello)

Riepilogo da portare al Notebook 02:

- **Pochissimo training (200):** il fine-tuning pesante è fuori discussione. La strada è un
  modello **pre-addestrato** usato in few-shot / zero-shot.
- **Scrittura a mano storica + formati caotici:** l'OCR classico (Tesseract) inciamperà sul
  corsivo e sui numeri romani. Un **VLM** (vision-language model) che "ragiona" sull'immagine
  ed estrae direttamente i due campi è molto più promettente — e può gestire le regole
  (escludi `Coll.`/`det.`, ignora `Dania` e lo sterco di mucca) via **prompt**.
- **AURC pesa:** dobbiamo produrre una confidence sensata. Idee: log-prob del modello,
  auto-consistenza (più campionamenti → accordo = confidence), o incrociare date/località.
- **Pipe multi-card:** possiamo usarle generosamente quando l'ordine è ambiguo — la metrica
  ci perdona.

**Prossima puntata (Notebook 02):** scegliamo tra un VLM open sul GPU Kaggle (es. Qwen2-VL)
e l'OCR classico come contro-prova, e ci costruiamo una strategia di confidence per l'AURC.

> Compito per te: apri 10 immagini a caso e prova a trascriverle *a mano*. Se fai fatica tu,
> capisci cosa stiamo chiedendo al modello. 🪲

---
# 🤖 Parte 02 — Baseline VLM zero-shot (Qwen3-VL)

L'EDA ci ha detto tutto: pochissimo training (200), scrittura a mano storica, formati
caotici. La risposta giusta non è un OCR addestrato da zero ma un **Vision-Language Model**
pre-addestrato che *guarda* l'immagine ed estrae direttamente i due campi, seguendo le
regole via **prompt** (escludi `Coll.`/`det.`, ignora `Dania` e lo sterco di mucca, usa
le pipe per i cartellini multipli).

**Piano:**
1. Carichiamo `Qwen2-VL-2B-Instruct` sul GPU di Kaggle (il 2B è veloce; c'è la variabile per
   passare al 7B se vuoi più qualità).
2. Prompt che codifica le regole → il modello risponde in **JSON**.
3. **Confidence** dai log-prob del modello (serve all'AURC: sapere quanto è sicuro).
4. **Validazione locale** su un pugno di esempi di train → confrontiamo la NED con il
   pavimento di **0.80** della baseline. *Prima di sparare su 3.300 immagini, verifichiamo
   che funzioni.*
5. `submission.csv` finale.

> ⚙️ **Prima di lanciare:** accendi la **GPU** (Settings → Accelerator → GPU T4) e, se la
> gara ha l'internet **OFF**, aggiungi Qwen2-VL dai *Kaggle Models* e punta `MODEL_ID` al
> path locale. Con internet ON, la cella qui sotto scarica tutto da sola.

### 7.1 Dipendenze e caricamento del modello

`qwen-vl-utils` gestisce il pre-processing delle immagini. Limitiamo la risoluzione massima
(`max_pixels`) per non far esplodere memoria e tempi: le etichette sono piccole, non serve il
4K.

In [ ]:
# Se l'internet è ON questa cella installa il necessario; se è OFF, salta e usa i Kaggle Models.
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
try:
    import qwen_vl_utils  # noqa
except Exception:
    _pip('qwen-vl-utils')
# Qwen2-VL richiede transformers recente
import transformers, importlib.metadata as im
try:
    if tuple(map(int, im.version('transformers').split('.')[:2])) < (4, 57):
        _pip('-U', 'transformers', 'accelerate')
except Exception:
    pass
print('transformers', transformers.__version__)

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- Modello: Qwen3-VL (multimodale, ufficiale QwenLM) ------------------------
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'   # taglie: 2B / 4B / 8B / 32B (Instruct)
# Se l'internet della gara e' OFF: aggiungi "Qwen 3 VL" dai Kaggle Models
# (framework Transformers) e punta qui il path locale, es.:
# MODEL_ID = '/kaggle/input/qwen-3-vl/transformers/4b-instruct/1'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Le GPU Kaggle (T4/P100) NON hanno bf16 nativo -> float16 e' molto piu' veloce
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
MAX_PIXELS = 1280 * 28 * 28                 # ~1.0MP: test di velocita' controllato
#   1280*28*28 ~1.0MP (veloce) | 2048*28*28 ~1.6MP | 4096*28*28 ~3.2MP (piu' memoria/tempo)

print('device:', DEVICE, '| dtype:', DTYPE)
# NIENTE device_map='auto': carichiamo TUTTO sulla GPU per evitare l'offload su CPU
# (anche pochi layer in RAM = inferenza lentissima).
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MAX_PIXELS)
print('Modello caricato:', MODEL_ID)

### 7.2 Il prompt (le regole della gara, in inglese per il modello)

Il prompt in inglese rende meglio, ma chiediamo di trascrivere **verbatim** (senza tradurre)
e in **JSON** rigido, così il parsing è banale. Ci infiliamo tutte le trappole viste nell'EDA.

In [ ]:
PROMPT = (
    "You are transcribing text from a museum specimen label "
    "(Danish dung beetle collection, late 1800s to present).\n"
    "Extract EXACTLY two fields, verbatim as written (keep original spelling and "
    "abbreviations, do NOT translate):\n"
    "- verbatimDate: the COLLECTION date as written (e.g. \"27.IV.2022\", \"18-7 1965\", \"IV.91\").\n"
    "- verbatimLocality: the COLLECTION locality as written (e.g. \"Dyrehaven\", \"Kb | Dyrehaven\").\n\n"
    "Rules:\n"
    "- If a field is not present on the label, output exactly \"" + MISSING_TOKEN + "\".\n"
    "- Do NOT include collector names (often prefixed \"Coll.\"), species names, or "
    "determination dates (prefixed \"det.\" or \"Tilg.\").\n"
    "- \"Dania\" is NOT a locality. Phrases meaning \"in cow dung\" (e.g. \"i kogøding\") are NOT a locality.\n"
    "- If information is split across multiple cards, separate the pieces with \" | \".\n\n"
    "Return ONLY a JSON object, nothing else: "
    "{\"verbatimDate\": \"...\", \"verbatimLocality\": \"...\"}"
)
print(PROMPT)

### 7.3 Funzione di inferenza + confidence

Estraiamo la risposta e, con `compute_transition_scores`, la **probabilità media** dei token
generati: è la nostra confidence per l'AURC. Se qualcosa va storto nel parsing, ripieghiamo
su `MISSING` con confidence bassa (meglio dire "non so" che sparare a caso).

In [ ]:
import json, re

# DIAGNOSI VELOCITA': con False la generate NON calcola output_scores (confidence fissa).
# Serve a isolare quanto pesa la macchina della confidence. Rimetti True per l'AURC.
USE_CONFIDENCE = True

def _parse_json(text):
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            d = json.loads(m.group(0))
            return str(d.get('verbatimDate', MISSING_TOKEN)), str(d.get('verbatimLocality', MISSING_TOKEN))
        except Exception:
            pass
    return MISSING_TOKEN, MISSING_TOKEN

@torch.no_grad()
def vlm_transcribe(image):
    "Ritorna (verbatimDate, verbatimLocality, confidence in (0,1])."
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    n_in = inputs.input_ids.shape[1]

    if USE_CONFIDENCE:
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)
        gen_ids = out.sequences[:, n_in:]
        try:
            trans = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
            conf = float(torch.exp(trans[0][trans[0].isfinite()].mean()).clamp(0, 1))
        except Exception:
            conf = 0.5
    else:
        # FAST PATH: nessun output_scores -> molto piu' leggero in memoria/tempo
        gen_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)[:, n_in:]
        conf = 0.5

    answer = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    date, loc = _parse_json(answer)
    return date, loc, conf

### 7.4 Sanity check: funziona meglio del pavimento (0.80)?

Regola d'oro: **non lanciare mai un loop da ore senza aver testato su pochi esempi.** Giriamo
il VLM su un piccolo campione di train e misuriamo la NED con la *nostra* metrica. Se non
scende ben sotto 0.80, c'è un bug o un problema di prompt — meglio scoprirlo ora.

In [ ]:
from tqdm.auto import tqdm

N_CHECK = 20   # tienilo piccolo: serve solo a verificare che la pipeline funzioni
check = train.sample(min(N_CHECK, len(train)), random_state=7).reset_index(drop=True)

rows = []
for _, r in tqdm(check.iterrows(), total=len(check), desc='sanity VLM'):
    img = load_image(r['image_file'])
    d, l, c = vlm_transcribe(img)
    rows.append({'image_file': r['image_file'], 'verbatimDate': d, 'verbatimLocality': l})
pred_check = pd.DataFrame(rows)

truth_check = check[['image_file', 'verbatimDate', 'verbatimLocality']]
d_ned, l_ned, avg_ned = score_df(pred_check, truth_check)
print('VLM zero-shot su %d esempi:' % len(check))
print('  NED date : %.3f' % d_ned)
print('  NED loc  : %.3f' % l_ned)
print('  NED avg  : %.3f   (baseline MISSING = 0.80)' % avg_ned)

# affianca predizioni e verità per un'ispezione a occhio
comp = truth_check.merge(pred_check, on='image_file', suffixes=('_true', '_pred'))
comp[['image_file','verbatimDate_true','verbatimDate_pred','verbatimLocality_true','verbatimLocality_pred']]

### 7.5 Inferenza sul test set → `submission.csv`

Se il sanity check convince, lancia l'inferenza completa. **Parti con `SMOKE_TEST = True`**
(solo 20 immagini) per stimare i tempi; poi metti `False` per il giro completo sulle ~3.300.

> A ~1 img/s il full run sono ~1 ora: assicurati che la GPU sia accesa e il notebook committato
> in modalità *Save & Run All*.

In [ ]:
SMOKE_TEST = True    # <-- metti False per l'inferenza completa sul test set

test_subset = test.head(20) if SMOKE_TEST else test
print('Immagini da processare:', len(test_subset), '(SMOKE_TEST =', SMOKE_TEST, ')')

results = []
for _, r in tqdm(test_subset.iterrows(), total=len(test_subset), desc='VLM test'):
    img = load_image(r['image_file'])
    if img is None:
        d, l, c = MISSING_TOKEN, MISSING_TOKEN, 0.1
    else:
        d, l, c = vlm_transcribe(img)
    # niente stringhe vuote: Kaggle non le accetta
    d = d.strip() or MISSING_TOKEN
    l = l.strip() or MISSING_TOKEN
    results.append({
        'image_file': r['image_file'],
        'verbatimDate': d, 'verbatimDate_confidence': round(c, 4),
        'verbatimLocality': l, 'verbatimLocality_confidence': round(c, 4),
    })

vlm_sub = pd.DataFrame(results)
vlm_sub.head(10)

In [ ]:
# Scrivi la submission SOLO se hai fatto il giro completo (non sullo smoke test)
if not SMOKE_TEST and len(vlm_sub) == len(test):
    assert not vlm_sub.isna().any().any(), 'ci sono NaN!'
    assert (vlm_sub[['verbatimDate','verbatimLocality']] != '').all().all(), 'celle vuote!'
    vlm_sub.to_csv('submission.csv', index=False)
    print('submission.csv scritto:', vlm_sub.shape)
else:
    print('SMOKE_TEST attivo (o test incompleto): submission NON scritta. '
          'Metti SMOKE_TEST=False e rilancia per generarla.')

### 7.6 Dove andare da qui (Notebook 03?)

Quando avrai i numeri del VLM zero-shot, i prossimi upgrade in ordine di rapporto
sforzo/beneficio:

1. **Post-processing a regole** — pulire l'output del modello (togliere `Coll.`/`det.`
   sfuggiti, normalizzare le pipe): quasi gratis, spesso qualche punto di NED.
2. **Few-shot nel prompt** — 2-3 esempi (immagine + risposta) dai 200 di train per "ancorare"
   il formato. Migliora soprattutto date romane e parziali.
3. **Confidence più furba per l'AURC** — auto-consistenza: campiona 2-3 volte, se le risposte
   concordano alza la confidence; separa la confidence di data e località.
4. **Modello più grande** — passa al 7B se il 2B non basta e i tempi lo consentono.

> Compito: guarda la tabella di confronto del §7.4 e trova **un** caso in cui il modello
> sbaglia in modo *sistematico* (es. include sempre il collezionatore). Quello ci dice quale
> regola di post-processing scrivere per prima. 🪲

## 8. Il "leaderboard casalingo": validazione completa su train (NED + AURC)

I 20 esempi del sanity servono a vedere se la pipeline respira. Per **decidere** serve un
numero stabile e fedele alla leaderboard, calcolato su **tutti i 200 di train** (~15 min a
~4-5 s/img). Questa cella misura tre cose:

1. **NED** per campo e media → quanto trascriviamo bene.
2. **AURC** con le confidence del modello → quanto la confidence sa *ordinare* i casi.
3. **Calibrazione** → i casi ad alta confidence hanno davvero NED più basso? (È ciò che l'AURC premia.)

D'ora in poi ogni modifica (risoluzione, few-shot, modello) si giudica **qui**, non sulla leaderboard.

In [ ]:
from tqdm.auto import tqdm

# Assicurati di avere USE_CONFIDENCE = True (serve una confidence vera per l'AURC).
val = train.reset_index(drop=True)
recs = []
for _, r in tqdm(val.iterrows(), total=len(val), desc='val 200'):
    img = load_image(r['image_file'])
    d, l, c = vlm_transcribe(img)
    nd = ned_single(d, r['verbatimDate'], True)
    nl = ned_single(l, r['verbatimLocality'], False)
    recs.append({
        'image_file': r['image_file'],
        'true_date': r['verbatimDate'], 'pred_date': d, 'ned_date': nd,
        'true_loc': r['verbatimLocality'], 'pred_loc': l, 'ned_loc': nl,
        'ned': (nd + nl) / 2, 'conf': c,
    })
val_res = pd.DataFrame(recs)

print('=== NED (piu basso = meglio) ===')
print('  date: %.3f | loc: %.3f | avg: %.3f   (baseline MISSING = 0.80)'
      % (val_res.ned_date.mean(), val_res.ned_loc.mean(), val_res.ned.mean()))

print('\n=== AURC (usa la confidence; proxy della leaderboard) ===')
print('  AURC locale: %.3f' % aurc(val_res['ned'].values, val_res['conf'].values))
# riferimento: AURC con confidence casuale (quanto guadagna l'ordinamento?)
import numpy as np
rng = np.random.default_rng(0)
print('  AURC con confidence random: %.3f' % aurc(val_res['ned'].values, rng.random(len(val_res))))

print('\n=== Calibrazione (vogliamo ALTA << BASSA) ===')
med = val_res.conf.median()
print('  NED media | conf ALTA: %.3f  vs  conf BASSA: %.3f'
      % (val_res[val_res.conf >= med].ned.mean(), val_res[val_res.conf < med].ned.mean()))

print('\n=== Dove perdiamo piu punti ===')
val_res.sort_values('ned', ascending=False).head(12)[
    ['image_file','true_date','pred_date','true_loc','pred_loc','ned','conf']]